# Measurement-based quantum reservoirs
Install the source package with `.[docs,graphix,validation]` after PhotoGraphiQ.
All parameters below are fixed; only ridge readouts are trained. CV Gaussian Tier B
observables are efficiently classically simulable. No quantum advantage is claimed.
A stateful reservoir transfers its full surviving state. `WindowedMBQELM` instead
resets for each explicit trailing window: its history is supplied classically.

In [ ]:
import importlib.util

import matplotlib.pyplot as plt
import numpy as np

from cv_mb_qrc.reservoirs import CVConfig, CVMBReservoir, GraphixMBReservoir, QubitConfig
from cv_mb_qrc.reservoirs.benchmarks import capacity_targets, metrics, select_readout
from cv_mb_qrc.reservoirs.temporal import chronological_splits

SEED = 7
DATA_SEED = 1729
WASHOUT = 20
DELAY = 3
config = CVConfig(
    seed=SEED,
    memory_modes=2,
    squeezing=0.3,
    coupling=0.2,
    transmissivity=0.8,
    tier="B",
    evolution="unconditional",
)
inputs = np.random.default_rng(DATA_SEED).uniform(-1, 1, 360)
cv = CVMBReservoir(config)
print(cv.run_sequence(inputs[:5]).features)
print(cv.summary())

In [ ]:
if importlib.util.find_spec("graphix"):
    qubit = GraphixMBReservoir(QubitConfig(seed=SEED))
    print(qubit.run_sequence(inputs[:5]).features)
    if importlib.util.find_spec("mentpy"):
        from cv_mb_qrc.reservoirs.mentpy_backend import compare_wire

        print(compare_wire(np.array([1, 1j]) / np.sqrt(2)))
else:
    print("Install cv-mb-qrc[graphix] to execute the optional qubit example.")

## Leakage-safe memory and nonlinear targets
Each chronological split gets its own reset, delay targets, and washout.
Legendre targets use independent uniform inputs on [-1,1]. Training statistics
are never fitted on validation or test data. The short test partition makes these
functionality checks, not reliable capacity estimates.

In [ ]:
splits = chronological_splits(len(inputs), gap=4, washout=WASHOUT)
data = {}
for name, indices in splits.items():
    u = inputs[indices]
    features = CVMBReservoir(config).run_sequence(u).features
    targets, names = capacity_targets(u, DELAY)
    data[name] = (u[WASHOUT:], features[WASHOUT:], targets[WASHOUT:])
scores = {}
for j, name in enumerate(names):
    train = (*data["train"][:2], data["train"][2][:, j])
    validation = (*data["validation"][:2], data["validation"][2][:, j])
    readout = select_readout(train, validation)
    predicted = readout.predict(*data["test"][:2])
    scores[name] = metrics(data["test"][2][:, j], predicted)["r2"]
print(
    "Nonlinear degree-two scores:", {k: v for k, v in scores.items() if not k.startswith("linear")}
)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, DELAY + 1), [scores[f"linear_{k}"] for k in range(1, DELAY + 1)], "o-")
ax.set(xlabel="Delay", ylabel="Raw test R?", title="Single-seed memory demonstration")
plt.show()

## Conditional trajectories and reproducibility
`shots` counts independently persistent quantum trajectories. Features use exact
conditional-state moments inside each trajectory, so this is not a detector-shot
budget for estimating all output observables. Ensemble covariance includes the
between-trajectory means. The experiment CLI uses ten independent reservoir seeds,
retains negative R? and raw arrays, and reports bootstrap confidence intervals.
See `experiments/measurement_based_reservoir/README.md` for the full commands.

In [ ]:
sampled = CVMBReservoir(config).run_sequence(inputs[:3], shots=32)
print(sampled.estimator, sampled.features.shape)
assert np.isfinite(sampled.features).all()